# A/B Test Analysis — mistral:7b-instruct Prompt Variant Comparison
**IDS 568 Final Project · Sreesahithi Gundapaneni (sgund12)**

This notebook loads the simulation results from `logs/ab_test_results.json` and provides
a detailed statistical walkthrough of the A/B test for EXP-001.

In [ ]:
import json
import numpy as np
import scipy.stats as stats
import matplotlib.pyplot as plt
import pandas as pd

# Load results
with open('../logs/ab_test_results.json') as f:
    results = json.load(f)

df = pd.DataFrame(results)
print('A/B Test Results Summary')
print('=' * 50)
df[['metric','a_mean','b_mean','diff','ci_low','ci_high','p_value','significant']]

In [ ]:
# Visualize confidence intervals
fig, ax = plt.subplots(figsize=(10, 5), facecolor='#0f1117')
ax.set_facecolor('#1a1d27')

metrics = [r['metric'] for r in results]
diffs   = [r['diff'] for r in results]
ci_lows = [r['ci_low'] for r in results]
ci_highs= [r['ci_high'] for r in results]
colors  = ['#00d4aa' if r['favors_b'] else '#ff4757' for r in results]

y = range(len(metrics))
ax.barh(y, diffs,
        xerr=[[d-l for d,l in zip(diffs,ci_lows)],
              [h-d for d,h in zip(diffs,ci_highs)]],
        color=colors, alpha=0.8, height=0.5, capsize=5)
ax.axvline(0, color='white', linestyle='--', linewidth=1)
ax.set_yticks(list(y))
ax.set_yticklabels(metrics, color='#e0e0e0')
ax.set_xlabel('Difference (B - A)', color='#e0e0e0')
ax.set_title('95% Confidence Intervals for Metric Differences (B - A)', color='white', fontweight='bold')
ax.tick_params(colors='#e0e0e0')
for spine in ax.spines.values():
    spine.set_color('#2a2d3a')
plt.tight_layout()
plt.show()
print('Green = favors Model B (challenger)')
print('Red   = favors Model A (baseline)')

In [ ]:
# Power analysis verification
from scipy.stats import norm

p1 = 0.72   # Model A baseline groundedness
mde = 0.05  # minimum detectable effect
p2 = p1 + mde
alpha = 0.05
power = 0.80

pooled = (p1 + p2) / 2
z_alpha = norm.ppf(1 - alpha/2)
z_beta  = norm.ppf(power)

n = ((z_alpha * np.sqrt(2 * pooled * (1-pooled)) +
      z_beta  * np.sqrt(p1*(1-p1) + p2*(1-p2)))**2) / (mde**2)

print(f'Power Analysis for Primary Metric (Groundedness)')
print(f'Baseline rate (p1):     {p1}')
print(f'Target rate (p2):       {p2}')
print(f'MDE:                    {mde}')
print(f'Alpha (two-tailed):     {alpha}')
print(f'Desired power:          {power}')
print(f'Required n per group:   {int(np.ceil(n))}')
print(f'\nNote: Simulation uses n=500 for demonstration.')
print(f'In production, run until n={int(np.ceil(n))} per arm is reached.')

## Decision

Based on the statistical analysis:
- Groundedness improvement of **+0.064** is statistically significant (p < 0.001)
- Latency increase of **+0.334s** is within acceptable bounds (mean P50 still under 1s)
- Success rate difference is **not significant** (p = 0.727)

**Recommendation: SHIP Model B** (enhanced grounding system prompt)

See `docs/recommendation-memo.md` for the full decision memo.